In [1]:
import sys, re
import pandas as pd
sys.path.insert(0, "..")
import spacy
from rapidfuzz import process, fuzz
import fitz
from src.preprocessing.segmenter import segment_document

nlp = spacy.load("en_core_web_lg")
df_qs = pd.read_csv("../data/qs_rankings_2025.csv")
qs_names = df_qs['university_name'].tolist()
print(f"spaCy chargé — QS DB : {len(qs_names)} universités")

spaCy chargé — QS DB : 62 universités


In [3]:
# Abréviations connues → nom complet dans la DB QS
ALIASES = {
    "MIT":   "Massachusetts Institute of Technology",
    "UCL":   "UCL",
    "LSE":   "London School of Economics",
    "EPFL":  "EPFL",
    "NYU":   "New York University",
    "UCLA":  "University of California Los Angeles",
    "UCSD":  "University of California San Diego",
    "UCB":   "University of California Berkeley",
    "SEMO":  "Southeast Missouri State University",
}
print("Aliases chargés :", len(ALIASES))

Aliases chargés : 9


In [4]:
from dataclasses import dataclass
from typing import Optional

@dataclass
class UniversityResult:
    detected_name:   Optional[str]
    qs_rank:         Optional[int]
    qs_match_name:   Optional[str]
    match_score:     float
    confidence:      float
    degree_level:    str

def extract_university(text: str,
                       degree_level: str = "unknown") -> UniversityResult:
    if not text or len(text.strip()) < 10:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 1. Résoudre les alias d'abord
    for alias, full_name in ALIASES.items():
        if re.search(r'\b' + re.escape(alias) + r'\b', text):
            row = df_qs[df_qs['university_name'] == full_name]
            rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            return UniversityResult(
                detected_name=full_name,
                qs_rank=rank,
                qs_match_name=full_name,
                match_score=1.0,
                confidence=0.95,
                degree_level=degree_level
            )

    # 2. spaCy NER — détecter les entités ORG
    doc = nlp(text[:2000])
    org_entities = [ent.text for ent in doc.ents if ent.label_ == "ORG"]

    if not org_entities:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 3. Fuzzy match chaque ORG contre la DB QS
    best_name, best_rank, best_score, best_match = None, None, 0.0, None

    for org in org_entities:
        result = process.extractOne(
            org, qs_names,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=60
        )
        if result and result[1] > best_score:
            best_score = result[1]
            best_match = result[0]
            row = df_qs[df_qs['university_name'] == best_match]
            best_rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            best_name = org

    if best_score == 0:
        return UniversityResult(
            detected_name=org_entities[0] if org_entities else None,
            qs_rank=None,
            qs_match_name=None,
            match_score=0.0,
            confidence=0.10,
            degree_level=degree_level
        )

    confidence = round(best_score / 100, 2)
    return UniversityResult(
        detected_name=best_name,
        qs_rank=best_rank,
        qs_match_name=best_match,
        match_score=round(best_score / 100, 2),
        confidence=confidence,
        degree_level=degree_level
    )

print("Extracteur université défini.")

Extracteur université défini.


In [5]:
# CV Esra — NYU (BSc) + University of Chester (MSc)
doc = fitz.open("../data/raw/cvs/257264722 CV.pdf")
text_cv = "".join(p.get_text() for p in doc)
seg = segment_document(text_cv)
edu_text = seg.sections.get("EDUCATION", "")

print("=== EDUCATION TEXT ===")
print(edu_text[:500])
print()

result = extract_university(edu_text, degree_level="bsc")
print(f"Détecté    : {result.detected_name}")
print(f"QS match   : {result.qs_match_name}")
print(f"QS rank    : {result.qs_rank}")
print(f"Score      : {result.match_score}")
print(f"Confiance  : {result.confidence}")

=== EDUCATION TEXT ===
Master of Science in Cybersecurity - Summer 2024
New York University, New York

Bachelor of Science in Software Engineering - Spring 2017
AlHosn University, Abu Dhabi, UAE

Détecté    : New York University
QS match   : New York University
QS rank    : 32
Score      : 1.0
Confiance  : 1.0


In [6]:
# Test séparé BSc vs MSc
print("=== TEST BSc — AlHosn University (non classée QS) ===")
bsc_text = "Bachelor of Science in Software Engineering - Spring 2017\nAlHosn University, Abu Dhabi, UAE"
result_bsc = extract_university(bsc_text, degree_level="bsc")
print(f"Détecté    : {result_bsc.detected_name}")
print(f"QS rank    : {result_bsc.qs_rank}")
print(f"Confiance  : {result_bsc.confidence}")
print(f"→ Attendu  : qs_rank=None (CL-04 — non classée)")

print("\n=== TEST MSc — New York University ===")
msc_text = "Master of Science in Cybersecurity - Summer 2024\nNew York University, New York"
result_msc = extract_university(msc_text, degree_level="msc")
print(f"Détecté    : {result_msc.detected_name}")
print(f"QS rank    : {result_msc.qs_rank}")
print(f"Confiance  : {result_msc.confidence}")
print(f"→ Attendu  : qs_rank=32, confiance=1.0")

=== TEST BSc — AlHosn University (non classée QS) ===
Détecté    : AlHosn University
QS rank    : 58
Confiance  : 0.79
→ Attendu  : qs_rank=None (CL-04 — non classée)

=== TEST MSc — New York University ===
Détecté    : New York University
QS rank    : 32
Confiance  : 1.0
→ Attendu  : qs_rank=32, confiance=1.0


In [8]:
import re

def split_bsc_msc(education_text: str) -> dict:
    """Sépare le texte EDUCATION en BSc et MSc."""
    bsc_keywords = r'(?i)(bachelor|b\.sc|bsc|b\.s\.|undergraduate|licence|licence)'
    msc_keywords = r'(?i)(master|m\.sc|msc|m\.s\.|postgraduate|magistere)'

    lines = education_text.split('\n')
    bsc_lines, msc_lines = [], []
    current = None

    for line in lines:
        if re.search(msc_keywords, line):
            current = 'msc'
        elif re.search(bsc_keywords, line):
            current = 'bsc'
        if current == 'bsc':
            bsc_lines.append(line)
        elif current == 'msc':
            msc_lines.append(line)

    return {
        "bsc": '\n'.join(bsc_lines),
        "msc": '\n'.join(msc_lines)
    }

# Test sur Esra
parts = split_bsc_msc(edu_text)
print("BSc text :", parts['bsc'])
print("\nMSc text :", parts['msc'])

bsc = extract_university(parts['bsc'], degree_level="bsc")
msc = extract_university(parts['msc'], degree_level="msc")

print(f"\nBSc → {bsc.detected_name} | rank={bsc.qs_rank} | conf={bsc.confidence}")
print(f"MSc → {msc.detected_name} | rank={msc.qs_rank} | conf={msc.confidence}")
print("\n→ Attendu : BSc=AlHosn rank=None, MSc=NYU rank=32")


BSc text : Bachelor of Science in Software Engineering - Spring 2017
AlHosn University, Abu Dhabi, UAE

MSc text : Master of Science in Cybersecurity - Summer 2024
New York University, New York


BSc → AlHosn University | rank=58 | conf=0.79
MSc → New York University | rank=32 | conf=1.0

→ Attendu : BSc=AlHosn rank=None, MSc=NYU rank=32


In [9]:
def extract_university_v2(text: str,
                           degree_level: str = "unknown") -> UniversityResult:
    if not text or len(text.strip()) < 10:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 1. Alias
    for alias, full_name in ALIASES.items():
        if re.search(r'\b' + re.escape(alias) + r'\b', text):
            row = df_qs[df_qs['university_name'] == full_name]
            rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            return UniversityResult(full_name, rank, full_name,
                                    1.0, 0.95, degree_level)

    # 2. spaCy NER
    doc = nlp(text[:2000])
    org_entities = [ent.text for ent in doc.ents if ent.label_ == "ORG"]

    if not org_entities:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 3. Fuzzy match avec seuil élevé à 85
    best_name, best_rank, best_score, best_match = None, None, 0.0, None

    for org in org_entities:
        result = process.extractOne(
            org, qs_names,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=85          # ← seuil relevé de 60 à 85
        )
        if result and result[1] > best_score:
            best_score = result[1]
            best_match = result[0]
            row = df_qs[df_qs['university_name'] == best_match]
            best_rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            best_name = org

    # 4. Si aucun match suffisant → université non classée
    if best_score == 0:
        return UniversityResult(
            detected_name=org_entities[0],
            qs_rank=None,            # ← non classée QS
            qs_match_name=None,
            match_score=0.0,
            confidence=0.10,         # confiance basse = signal pour LLM fallback
            degree_level=degree_level
        )

    return UniversityResult(
        detected_name=best_name,
        qs_rank=best_rank,
        qs_match_name=best_match,
        match_score=round(best_score / 100, 2),
        confidence=round(best_score / 100, 2),
        degree_level=degree_level
    )

# Retester
bsc = extract_university_v2(parts['bsc'], degree_level="bsc")
msc = extract_university_v2(parts['msc'], degree_level="msc")

print(f"BSc → {bsc.detected_name} | rank={bsc.qs_rank} | conf={bsc.confidence}")
print(f"MSc → {msc.detected_name} | rank={msc.qs_rank} | conf={msc.confidence}")
print()
print("BSc attendu : AlHosn University | rank=None | conf=0.10  ✓")
print("MSc attendu : New York University | rank=32  | conf=1.0   ✓")

BSc → Bachelor of Science in Software Engineering - Spring | rank=None | conf=0.1
MSc → New York University | rank=32 | conf=1.0

BSc attendu : AlHosn University | rank=None | conf=0.10  ✓
MSc attendu : New York University | rank=32  | conf=1.0   ✓


In [10]:
# Pattern regex pour capturer le nom d'université directement
UNIVERSITY_REGEX = re.compile(
    r'(?:university|université|institute|college|school|academy|polytechnic)'
    r'[^,\n]{0,60}',
    re.IGNORECASE
)

def extract_university_v3(text: str,
                           degree_level: str = "unknown") -> UniversityResult:
    if not text or len(text.strip()) < 10:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 1. Alias exacts
    for alias, full_name in ALIASES.items():
        if re.search(r'\b' + re.escape(alias) + r'\b', text):
            row = df_qs[df_qs['university_name'] == full_name]
            rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            return UniversityResult(full_name, rank, full_name,
                                    1.0, 0.95, degree_level)

    # 2. Regex — chercher les lignes contenant "University", "College" etc.
    candidates = []
    for line in text.split('\n'):
        line = line.strip()
        if UNIVERSITY_REGEX.search(line) and len(line) > 5:
            # Exclure les lignes qui sont clairement des diplômes
            if not re.search(r'(?i)bachelor|master|doctor|phd|b\.sc|m\.sc', line):
                candidates.append(line)

    # 3. spaCy NER comme fallback si regex ne trouve rien
    if not candidates:
        doc = nlp(text[:2000])
        candidates = [e.text for e in doc.ents if e.label_ == "ORG"]

    if not candidates:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 4. Fuzzy match contre DB QS avec seuil 85
    best_name, best_rank, best_score, best_match = None, None, 0.0, None

    for candidate in candidates:
        result = process.extractOne(
            candidate, qs_names,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=85
        )
        if result and result[1] > best_score:
            best_score = result[1]
            best_match = result[0]
            row = df_qs[df_qs['university_name'] == best_match]
            best_rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            best_name = candidate

    # 5. Université non classée QS
    if best_score == 0:
        detected = candidates[0] if candidates else None
        return UniversityResult(
            detected_name=detected,
            qs_rank=None,
            qs_match_name=None,
            match_score=0.0,
            confidence=0.10,
            degree_level=degree_level
        )

    return UniversityResult(
        detected_name=best_name,
        qs_rank=best_rank,
        qs_match_name=best_match,
        match_score=round(best_score / 100, 2),
        confidence=round(best_score / 100, 2),
        degree_level=degree_level
    )

# Test final
bsc = extract_university_v3(parts['bsc'], degree_level="bsc")
msc = extract_university_v3(parts['msc'], degree_level="msc")

print(f"BSc → {bsc.detected_name} | rank={bsc.qs_rank} | conf={bsc.confidence}")
print(f"MSc → {msc.detected_name} | rank={msc.qs_rank} | conf={msc.confidence}")
print()
print("BSc attendu : AlHosn University | rank=None | conf=0.10  ✓")
print("MSc attendu : New York University | rank=32  | conf=1.0  ✓")


BSc → AlHosn University, Abu Dhabi, UAE | rank=None | conf=0.1
MSc → New York University, New York | rank=None | conf=0.1

BSc attendu : AlHosn University | rank=None | conf=0.10  ✓
MSc attendu : New York University | rank=32  | conf=1.0  ✓


In [11]:
def clean_candidate(text: str) -> str:
    """Supprimer ville, pays, virgules après le nom d'université."""
    # Garder seulement jusqu'à la première virgule
    text = text.split(',')[0].strip()
    # Supprimer les suffixes parasites
    text = re.sub(r'\s+(Abu Dhabi|UAE|New York|London|Paris|Beijing).*$',
                  '', text, flags=re.IGNORECASE)
    return text.strip()

def extract_university_final(text: str,
                              degree_level: str = "unknown") -> UniversityResult:
    if not text or len(text.strip()) < 10:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 1. Alias exacts
    for alias, full_name in ALIASES.items():
        if re.search(r'\b' + re.escape(alias) + r'\b', text):
            row = df_qs[df_qs['university_name'] == full_name]
            rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            return UniversityResult(full_name, rank, full_name,
                                    1.0, 0.95, degree_level)

    # 2. Regex — lignes contenant "University", "College" etc. sans mots diplôme
    candidates = []
    for line in text.split('\n'):
        line = line.strip()
        if UNIVERSITY_REGEX.search(line) and len(line) > 5:
            if not re.search(
                r'(?i)\b(bachelor|master|doctor|phd|b\.sc|m\.sc|'
                r'degree|science|engineering|cybersecurity)\b', line):
                candidates.append(clean_candidate(line))

    # 3. Fallback spaCy NER
    if not candidates:
        doc = nlp(text[:2000])
        candidates = [clean_candidate(e.text)
                      for e in doc.ents if e.label_ == "ORG"]

    if not candidates:
        return UniversityResult(None, None, None, 0.0, 0.0, degree_level)

    # 4. Fuzzy match — token_set_ratio ignore les mots supplémentaires
    best_name, best_rank, best_score, best_match = None, None, 0.0, None

    for candidate in candidates:
        result = process.extractOne(
            candidate, qs_names,
            scorer=fuzz.token_set_ratio,   # ← token_set au lieu de token_sort
            score_cutoff=85
        )
        if result and result[1] > best_score:
            best_score = result[1]
            best_match = result[0]
            row = df_qs[df_qs['university_name'] == best_match]
            best_rank = int(row.iloc[0]['rank']) if len(row) > 0 else None
            best_name = candidate

    # 5. Non classée QS
    if best_score == 0:
        return UniversityResult(
            detected_name=candidates[0] if candidates else None,
            qs_rank=None,
            qs_match_name=None,
            match_score=0.0,
            confidence=0.10,
            degree_level=degree_level
        )

    return UniversityResult(
        detected_name=best_name,
        qs_rank=best_rank,
        qs_match_name=best_match,
        match_score=round(best_score / 100, 2),
        confidence=round(best_score / 100, 2),
        degree_level=degree_level
    )

# Test final
bsc = extract_university_final(parts['bsc'], degree_level="bsc")
msc = extract_university_final(parts['msc'], degree_level="msc")

print(f"BSc → {bsc.detected_name} | rank={bsc.qs_rank} | conf={bsc.confidence}")
print(f"MSc → {msc.detected_name} | rank={msc.qs_rank} | conf={msc.confidence}")
print()
print("BSc attendu : AlHosn University | rank=None | conf=0.10  ✓")
print("MSc attendu : New York University | rank=32  | conf=1.0  ✓")

BSc → AlHosn University | rank=None | conf=0.1
MSc → New York University | rank=32 | conf=1.0

BSc attendu : AlHosn University | rank=None | conf=0.10  ✓
MSc attendu : New York University | rank=32  | conf=1.0  ✓
